# Feature Generation & Organisation

This notebook loads all feature representations produced by Notebooks 1 and 2,
computes PCA baselines, and organises everything into a single dictionary
structure that Notebook 4 can consume directly.

**Environment:** `ml_env`

## Feature sets prepared

| Key | Description | Source |
|-----|-------------|--------|
| `raw` | All raw standardised features (288 or 137 if audio-only) | Notebook 1 |
| `pca_{k}` | Top k PCA components from raw features | Computed here |
| `slow_linear_{k}` | k slowest linear SFA features | Notebook 2 |
| `slow_deg2_{k}` | k slowest degree-2 polynomial SFA features | Notebook 2 |
| `slow_deg3_{k}` | k slowest degree-3 polynomial SFA features | Notebook 2 |

Where k ∈ {5, 10, 15, 20, 25, 30}.

## Targets prepared

| Key | Description | Task |
|-----|-------------|------|
| `y_reg_arousal` | Continuous median arousal | Regression |
| `y_reg_valence` | Continuous median valence | Regression |
| `y_clf_arousal` | Binary arousal class (median split) | Classification |
| `y_clf_valence` | Binary valence class (median split) | Classification |

In [1]:
import numpy as np
import os
import pickle
from sklearn.decomposition import PCA

FEATURES_DIR = "features"

## 1. Load Raw Features & Targets from Notebook 1

In [2]:
# Raw standardised features
X_raw = np.load(os.path.join(FEATURES_DIR, "raw_features.npy"))

# Participant IDs (for Group K-Fold / LOPO cross-validation)
participant_ids = np.load(os.path.join(FEATURES_DIR, "participant_ids.npy"))

# Regression targets
y_reg_arousal = np.load(os.path.join(FEATURES_DIR, "y_reg_arousal.npy"))
y_reg_valence = np.load(os.path.join(FEATURES_DIR, "y_reg_valence.npy"))

# Classification targets
y_clf_arousal = np.load(os.path.join(FEATURES_DIR, "y_clf_arousal.npy"))
y_clf_valence = np.load(os.path.join(FEATURES_DIR, "y_clf_valence.npy"))

# Audio-only subset (first 137 columns) — match what Notebook 2 used
# Set AUDIO_ONLY = False when you expand to the full feature set
AUDIO_ONLY = False
if AUDIO_ONLY:
    X_raw = X_raw[:, :137]
    feature_label = "audio"
else:
    feature_label = "all"

print(f"Raw features: {X_raw.shape}")
print(f"Participants:  {len(np.unique(participant_ids))} unique")
print(f"Targets:       arousal ({y_reg_arousal.shape[0]}), valence ({y_reg_valence.shape[0]})")

Raw features: (13066, 288)
Participants:  18 unique
Targets:       arousal (13066), valence (13066)


## 2. Compute PCA Baselines

PCA is computed here (not in Notebook 4) so that PCA features are stored
alongside SFA features in the same structure. This keeps Notebook 4 focused
purely on model training and evaluation.

In [3]:
component_dims = [5, 10, 15, 20, 25, 30]

pca_features = {}

for k in component_dims:
    pca = PCA(n_components=k)
    X_pca = pca.fit_transform(X_raw)
    pca_features[k] = X_pca

    explained = np.sum(pca.explained_variance_ratio_) * 100
    print(f"PCA {k:2d} components: shape={X_pca.shape}, variance explained={explained:.1f}%")

PCA  5 components: shape=(13066, 5), variance explained=28.8%
PCA 10 components: shape=(13066, 10), variance explained=41.7%
PCA 15 components: shape=(13066, 15), variance explained=48.6%
PCA 20 components: shape=(13066, 20), variance explained=54.4%
PCA 25 components: shape=(13066, 25), variance explained=59.3%
PCA 30 components: shape=(13066, 30), variance explained=63.3%


## 3. Load SFA Features from Notebook 2

Load all three SFA variants (slow linear, slow degree 2, slow degree 3)
at each output dimension. These were saved by Notebook 2 as `.npy` files.

In [4]:
sfa_variants = ["slow_linear", "slow_deg2", "slow_deg3"]

sfa_features = {variant: {} for variant in sfa_variants}

for variant in sfa_variants:
    for k in component_dims:
        fname = os.path.join(FEATURES_DIR, f"{variant}_{feature_label}_{k}.npy")
        if os.path.exists(fname):
            sfa_features[variant][k] = np.load(fname)
            print(f"Loaded: {variant}_{feature_label}_{k}.npy  shape={sfa_features[variant][k].shape}")
        else:
            print(f"WARNING: {fname} not found — run Notebook 2 first")

Loaded: slow_linear_all_5.npy  shape=(13066, 5)
Loaded: slow_linear_all_10.npy  shape=(13066, 10)
Loaded: slow_linear_all_15.npy  shape=(13066, 15)
Loaded: slow_linear_all_20.npy  shape=(13066, 20)
Loaded: slow_linear_all_25.npy  shape=(13066, 25)
Loaded: slow_linear_all_30.npy  shape=(13066, 30)
Loaded: slow_deg2_all_5.npy  shape=(13066, 5)
Loaded: slow_deg2_all_10.npy  shape=(13066, 10)
Loaded: slow_deg2_all_15.npy  shape=(13066, 15)
Loaded: slow_deg2_all_20.npy  shape=(13066, 20)
Loaded: slow_deg2_all_25.npy  shape=(13066, 25)
Loaded: slow_deg2_all_30.npy  shape=(13066, 30)
Loaded: slow_deg3_all_5.npy  shape=(13066, 5)
Loaded: slow_deg3_all_10.npy  shape=(13066, 10)
Loaded: slow_deg3_all_15.npy  shape=(13066, 15)
Loaded: slow_deg3_all_20.npy  shape=(13066, 20)
Loaded: slow_deg3_all_25.npy  shape=(13066, 25)
Loaded: slow_deg3_all_30.npy  shape=(13066, 30)


## 4. Assemble Feature Dictionary

All feature sets are collected into a single dictionary `feature_sets` where:
- Keys are descriptive names (e.g., `"raw"`, `"pca_10"`, `"slow_linear_20"`)
- Values are numpy arrays of shape `(n_samples, n_features)`

This is the single data structure that Notebook 4 will iterate over.

In [5]:
feature_sets = {}

# Raw features (full dimensionality)
feature_sets["raw"] = X_raw

# PCA features
for k in component_dims:
    feature_sets[f"pca_{k}"] = pca_features[k]

# SFA features (all three variants)
for variant in sfa_variants:
    for k in component_dims:
        if k in sfa_features[variant]:
            feature_sets[f"{variant}_{k}"] = sfa_features[variant][k]

# Targets dictionary
targets = {
    "regression": {
        "arousal": y_reg_arousal,
        "valence": y_reg_valence,
    },
    "classification": {
        "arousal": y_clf_arousal,
        "valence": y_clf_valence,
    },
}

print(f"Total feature sets: {len(feature_sets)}")
print(f"\nFeature sets available:")
for name, X in sorted(feature_sets.items()):
    print(f"  {name:25s}  shape={X.shape}")

Total feature sets: 25

Feature sets available:
  pca_10                     shape=(13066, 10)
  pca_15                     shape=(13066, 15)
  pca_20                     shape=(13066, 20)
  pca_25                     shape=(13066, 25)
  pca_30                     shape=(13066, 30)
  pca_5                      shape=(13066, 5)
  raw                        shape=(13066, 288)
  slow_deg2_10               shape=(13066, 10)
  slow_deg2_15               shape=(13066, 15)
  slow_deg2_20               shape=(13066, 20)
  slow_deg2_25               shape=(13066, 25)
  slow_deg2_30               shape=(13066, 30)
  slow_deg2_5                shape=(13066, 5)
  slow_deg3_10               shape=(13066, 10)
  slow_deg3_15               shape=(13066, 15)
  slow_deg3_20               shape=(13066, 20)
  slow_deg3_25               shape=(13066, 25)
  slow_deg3_30               shape=(13066, 30)
  slow_deg3_5                shape=(13066, 5)
  slow_linear_10             shape=(13066, 10)
  slow_linear_

## 5. Verification

Sanity checks to ensure all feature sets are consistent before passing to Notebook 4.

In [6]:
n_samples = X_raw.shape[0]
errors = []

for name, X in feature_sets.items():
    # Check sample count matches
    if X.shape[0] != n_samples:
        errors.append(f"{name}: expected {n_samples} samples, got {X.shape[0]}")

    # Check for NaNs
    nan_count = np.isnan(X).sum()
    if nan_count > 0:
        errors.append(f"{name}: contains {nan_count} NaN values")

    # Check for infinite values
    inf_count = np.isinf(X).sum()
    if inf_count > 0:
        errors.append(f"{name}: contains {inf_count} infinite values")

# Check targets match
for task, task_targets in targets.items():
    for target_name, y in task_targets.items():
        if y.shape[0] != n_samples:
            errors.append(f"target {task}/{target_name}: expected {n_samples} samples, got {y.shape[0]}")

# Check participant IDs match
if participant_ids.shape[0] != n_samples:
    errors.append(f"participant_ids: expected {n_samples}, got {participant_ids.shape[0]}")

if errors:
    print("ERRORS FOUND:")
    for e in errors:
        print(f"  - {e}")
else:
    print(f"All {len(feature_sets)} feature sets passed verification.")
    print(f"All targets verified ({n_samples} samples each).")
    print(f"Participant IDs verified ({len(np.unique(participant_ids))} participants).")

All 25 feature sets passed verification.
All targets verified (13066 samples each).
Participant IDs verified (18 participants).


## 6. Save Consolidated Data

Save everything as a single pickle file that Notebook 4 can load in one line.
This avoids repeating all the loading/organisation logic in Notebook 4.

In [7]:
output = {
    "feature_sets": feature_sets,
    "targets": targets,
    "participant_ids": participant_ids,
    "component_dims": component_dims,
    "sfa_variants": sfa_variants,
    "feature_label": feature_label,
}

output_path = os.path.join(FEATURES_DIR, "experiment_data.pkl")
with open(output_path, "wb") as f:
    pickle.dump(output, f)

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Saved: {output_path} ({file_size_mb:.1f} MB)")
print(f"\nContents:")
print(f"  feature_sets:    {len(feature_sets)} sets")
print(f"  targets:         regression (arousal, valence), classification (arousal, valence)")
print(f"  participant_ids: {len(np.unique(participant_ids))} participants")
print(f"\nNotebook 4 can load this with:")
print(f'  with open("{output_path}", "rb") as f:')
print(f'      data = pickle.load(f)')

Saved: features\experiment_data.pkl (71.1 MB)

Contents:
  feature_sets:    25 sets
  targets:         regression (arousal, valence), classification (arousal, valence)
  participant_ids: 18 participants

Notebook 4 can load this with:
  with open("features\experiment_data.pkl", "rb") as f:
      data = pickle.load(f)


## Summary

The `experiment_data.pkl` file contains everything Notebook 4 needs:

```python
data["feature_sets"]     # dict: name -> numpy array
data["targets"]          # dict: task -> target_name -> numpy array
data["participant_ids"]  # numpy array for Group K-Fold CV
data["component_dims"]   # [5, 10, 15, 20, 25, 30]
data["sfa_variants"]     # ["slow_linear", "slow_deg2", "slow_deg3"]
```

**Expected feature sets (when all SFA features are available):**
- 1 raw feature set
- 6 PCA feature sets (5, 10, 15, 20, 25, 30 components)
- 18 SFA feature sets (3 variants × 6 dimensions)
- **Total: 25 feature sets**